# Customer Migration to OneBill — Combined Pipeline

End-to-end migration of **Consumer accounts and their contacts** from the
legacy Voyager stack into OneBill. This notebook combines two previously
separate flows into a single coherent pipeline:

1. **Account source** — MySQL (`bi_custom_views.reporting_account_allColumns`).
   Provides the master account record: name, address, billing details,
   and the *account-level* contact (the primary person on the account).
2. **Contact source** — Microsoft Dataverse (Dynamics CRM, `contact` entity).
   Provides *additional contacts* attached to each account (Billing, Technical,
   Primary, Communication, etc.) along with their Dynamics contact-type
   classification.

Both data sources are joined on **accountnumber** (called `AccountCode` in
MySQL and `accountnumber` on the linked Dataverse account record). Each
OneBill `POST /subscriber` request therefore contains:

- One **account-level contact** built from the MySQL row (no `contactAttributes`).
- Zero or more **Dynamics contacts** appended after it, each carrying a
  `contactAttributes` block listing their assigned contact types via the
  multi-value custom attribute `Dynamics Contact Types` (id `160203`).

## Execution order

The cells below are written so the notebook can be executed top-to-bottom.
At a high level:

| Section | What happens |
|---|---|
| 1. Imports & config | Load credentials, set tunables |
| 2. Dataverse auth + FetchXML | Define how to talk to Dynamics |
| 3. Fetch contacts | Pull all ~47k Consumer contacts using keyset pagination |
| 4. Contact-type mapping | Translate Dynamics option-set codes to OneBill labels |
| 5. MySQL account query | Pull all Consumer accounts from BI views |
| 6. Index contacts by account | Build an O(1) lookup dict keyed by accountnumber |
| 7. OAuth token manager | Thread-safe bearer token cache for OneBill |
| 8. Payload builder | Compose the combined account + contacts JSON |
| 9. POST to OneBill | Single-request worker with profiling |
| 10. Migrate (parallel) | ThreadPoolExecutor over all accounts |
| 11. Run + results | Execute and inspect failures |


## 1. Imports and Environment

Loads all libraries needed across both data sources and the migration loop.
Credentials are read from a `.env` file — `override=True` ensures `.env`
values win over anything pre-existing in the shell environment, which
matters when re-running this notebook after rotating a secret.

In [49]:
# %pip install msal mysql-connector-python sqlalchemy python-dotenv

# --- Core / std lib ---
import os
import re
import json
import time
import logging
import threading
import urllib.parse
from datetime import datetime, timedelta
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from html import escape as xml_escape

# --- Third-party ---
import requests
import pandas as pd
from sqlalchemy import create_engine
from msal import ConfidentialClientApplication
from dotenv import load_dotenv

# override=True makes the file authoritative over the existing process env
load_dotenv(override=True)

2026-05-13 09:15:57,286 [WARNING] python-dotenv could not parse statement starting at line 1
2026-05-13 09:15:57,290 [WARNING] python-dotenv could not parse statement starting at line 5
2026-05-13 09:15:57,297 [WARNING] python-dotenv could not parse statement starting at line 11
2026-05-13 09:15:57,300 [WARNING] python-dotenv could not parse statement starting at line 14


True

## 2. Configuration

All tunables and external endpoints live in one place so the rest of the
notebook never needs to touch `os.environ`. The expected `.env` keys are:

| Variable | Purpose |
|---|---|
| `CRM_TENANT_ID` / `CRM_CLIENT_ID` / `CRM_CLIENT_SECRET` | Azure AD app for Dataverse |
| `CRM_ENVIRONMENT_URL` | Dataverse env URL, no trailing slash |
| `DB_USERNAME` / `DB_PASSWORD` / `DB_HOST` | MySQL access for the BI views |
| `CLIENT_ID` / `CLIENT_SECRET` / `API_USERNAME` / `API_PASSWORD` | OneBill OAuth2 password-grant credentials |

`MAX_WORKERS` controls migration concurrency. OneBill's sandbox has been
observed to throttle past ~20 concurrent writers — increase cautiously.

In [50]:
# --- Dataverse (Dynamics CRM) ---
CRM_TENANT_ID       = os.environ["CRM_TENANT_ID"]
CRM_CLIENT_ID       = os.environ["CRM_CLIENT_ID"]
CRM_CLIENT_SECRET   = os.environ["CRM_CLIENT_SECRET"]
CRM_ENVIRONMENT_URL = os.environ["CRM_ENVIRONMENT_URL"]

# --- MySQL (BI source) ---
BI_DATASTORE_URL = (
    f"mysql+mysqlconnector://{os.environ['DB_USERNAME']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}/bi_datastore"
)

BI_CURATED_VIEWS_URL = (
    f"mysql+mysqlconnector://{os.environ['DB_USERNAME']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}/bi_curated_views"
)

# --- OneBill (destination) ---
ONEBILL_BASE_URL    = "https://sandbox-sg.onebillsoftware.com"
ONEBILL_TOKEN_URL   = f"{ONEBILL_BASE_URL}/oauth/token"
ONEBILL_PROXY_ACCT  = os.environ["PROXY_ACCOUNT_NUMBER"]  # partner / proxy account header value

# --- Migration tunables ---
MAX_WORKERS         = 20
TOKEN_TTL_SECONDS   = 3500  # safety margin under the typical 3600s OAuth TTL

## 3. Logging

Every run writes a timestamped log file alongside the notebook so failures
can be re-investigated later. Both the file and the console get the same
output, which is helpful when running headlessly.

In [51]:
log_filename = f'migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_filename), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

## 4. Dataverse FetchXML Query

Pulls every active Consumer contact (`statecode = 0`, parent account's
`accountcategorycode = 1`, account has a non-null `accountnumber`).

A few details worth noting:

- **`<order attribute="contactid">`** is required for stable paging. Without
  it Dataverse cannot resolve the paging cookie reliably when `distinct` is
  in play, which previously caused the loop to return page 1 repeatedly.
- **`distinct` is omitted** — `parentcustomerid` is a single-value lookup
  so the inner join cannot produce duplicate contact rows.
- **`accountnumber` from the linked account** comes back aliased as
  `a_ce3f647c660e4a5c99cc3d631f23406d.accountnumber`. We rename it later.

The FetchXML is parameterised on `contactid > {last_seen}` rather than using
the FetchXML paging cookie — see `get_contacts` for why.

In [52]:
# {extra_condition} is filled in per-page with a `contactid gt <last_seen>` clause
FETCHXML_TEMPLATE = '''
<fetch version="1.0" output-format="xml-platform" mapping="logical" no-lock="false" count="5000">
  <entity name="contact">
    <attribute name="entityimage_url"/>
    <attribute name="fullname"/>
    <attribute name="emailaddress1"/>
    <attribute name="telephone1"/>
    <attribute name="contactid"/>
    <attribute name="vgr_contacttypes"/>
    <attribute name="mobilephone"/>
    <attribute name="firstname"/>
    <attribute name="lastname"/>
    <attribute name="vgr_contactcode"/>
    <attribute name="birthdate"/>
    <order attribute="contactid" descending="false"/>
    <filter type="and">
      <condition attribute="vgr_contactcode" operator="not-like" value="BILLING%"/>
      {extra_condition}
    </filter>
    <link-entity name="account" from="accountid" to="parentcustomerid" link-type="inner" alias="AccountCode">
      <attribute name="accountnumber"/>
      <filter type="and">
        <condition attribute="accountcategorycode" operator="eq" value="1"/>
        <condition attribute="accountnumber" operator="not-null"/>
      </filter>
    </link-entity>
  </entity>
</fetch>
'''.strip()

# The aliased column name Dataverse returns for accountnumber via the link-entity
# ACCOUNTNUMBER_COL = "a_ce3f647c660e4a5c99cc3d631f23406d.accountnumber"
ACCOUNTNUMBER_COL = "AccountCode"  # using SQL alias in the view instead of the default Dataverse one

## 5. Dataverse Authentication

Standard OAuth2 client-credentials flow against Azure AD. The token is
valid for ~1 hour and scoped to the Dataverse environment. We only need it
once per notebook run since the contact fetch typically completes in under
a minute.

In [53]:
def get_dataverse_token() -> str:
    """Acquire an OAuth2 bearer token for Dataverse via client-credentials."""
    app = ConfidentialClientApplication(
        client_id=CRM_CLIENT_ID,
        client_credential=CRM_CLIENT_SECRET,
        authority=f"https://login.microsoftonline.com/{CRM_TENANT_ID}",
    )
    result = app.acquire_token_for_client(scopes=[f"{CRM_ENVIRONMENT_URL}/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"Token acquisition failed: {result.get('error_description')}")
    return result["access_token"]

## 6. Fetch Contacts with Keyset Pagination

Dataverse caps each FetchXML response at 5,000 records. The official way
to page is via the `paging-cookie` attribute, but with `distinct` or a
link-entity join the cookie can fail to advance silently — the server
quietly returns page 1 forever. To avoid that fragility we use **keyset
pagination** instead:

1. Order by `contactid` ascending (stable, unique, indexed).
2. On the first page, request without any extra filter — get records
   1..5000.
3. On each subsequent page, inject a `contactid > <last contactid seen>`
   filter and re-issue the request.
4. Stop when a response returns fewer than 5,000 rows.

This is more robust than cookie-based paging:
- Immune to the `distinct`/link-entity cookie bug.
- Naturally idempotent — re-running from the last seen `contactid` resumes
  cleanly.
- URL length stays small (no embedded cookie blob).

In [54]:
def get_contacts(token: str, max_pages: int = 50) -> pd.DataFrame:
    """
    Fetch all active Consumer contacts from Dataverse using keyset pagination
    on `contactid`. Returns a DataFrame with the link-entity columns left
    in their aliased form (e.g. `a_ce3f647c660e4a5c99cc3d631f23406d.accountnumber`).
    """
    headers = {
        "Authorization":   f"Bearer {token}",
        "OData-MaxVersion": "4.0",
        "OData-Version":    "4.0",
        "Accept":           "application/json",
        "Prefer":           'odata.maxpagesize=5000',
    }

    all_records: list[dict] = []
    last_contactid: str | None = None
    page = 1

    while True:
        # Inject the keyset cursor on every page except the first
        if last_contactid is None:
            extra_condition = ""
        else:
            extra_condition = (
                f'<condition attribute="contactid" operator="gt" value="{last_contactid}"/>'
            )

        fetch = FETCHXML_TEMPLATE.format(extra_condition=extra_condition)
        url = f"{CRM_ENVIRONMENT_URL}/api/data/v9.2/contacts?fetchXml={urllib.parse.quote(fetch)}"

        response = requests.get(url, headers=headers, timeout=60)
        response.raise_for_status()
        records = response.json().get("value", [])

        # Empty response = nothing left to fetch
        if not records:
            break

        all_records.extend(records)
        new_last = records[-1]["contactid"]
        logger.info(
            f"Contacts page {page}: fetched {len(records):,} "
            f"(last contactid={new_last}, total so far: {len(all_records):,})"
        )

        # Short page = final page
        if len(records) < 5000:
            break

        # Safety: if the cursor doesn't advance something has gone wrong upstream
        if new_last == last_contactid:
            logger.warning("contactid did not advance — stopping to avoid infinite loop")
            break

        last_contactid = new_last
        page += 1

        if page > max_pages:
            logger.warning(f"Hit max_pages safety limit ({max_pages})")
            break

    df = pd.DataFrame(all_records)
    logger.info(f"Done — {len(df):,} contacts loaded")
    return df

## 7. Run the Contact Fetch

Pulls all contacts into `df_crm_contacts`.

In [55]:
dataverse_token = get_dataverse_token()
df_crm_contacts = get_contacts(dataverse_token)
df_crm_contacts = df_crm_contacts.drop(columns=['@odata.etag', 'contactid', 'fullname'], axis=1)
df_crm_contacts = df_crm_contacts.rename(
    columns=
    {
        "vgr_contactcode": "ContactCode",
        "vgr_contacttypes": "ContactType",
        "telephone1": "PhoneWork",
        "mobilephone": "PhoneMobile",
        "emailaddress1": "EmailAddresses",
        "firstname": "FirstName",
        "lastname": "LastName",
        "AccountCode.accountnumber": "AccountCode",
        "birthdate": "DateOfBirth"
    })


df_crm_contacts.head()

2026-05-13 09:16:01,763 [INFO] Contacts page 1: fetched 5,000 (last contactid=d73c6341-46ad-ec11-9840-002248d39307, total so far: 5,000)
2026-05-13 09:16:03,499 [INFO] Contacts page 2: fetched 854 (last contactid=0ef65275-9b8b-4854-b0f0-f2f84858ab15, total so far: 5,854)
2026-05-13 09:16:03,503 [INFO] Done — 5,854 contacts loaded


,PhoneMobile,LastName,ContactCode,FirstName,EmailAddresses,AccountCode,PhoneWork,ContactType,DateOfBirth
0,0212141547,Sibbe,C-00100775,Judy,sibbe@actrix.co.nz,99993083,NaN,NaN,NaN
1,0211408346,Dunne,C-00092022,Kathryn,katiedunne30@gmail.com,99978495,NaN,NaN,NaN
2,021951128,Choi,C-00092943,Tina,by09010901@hotmail.com,99969820,NaN,NaN,NaN
3,+64272934772,Quinn,C-00101851,Laura,laurahopequinn@gmail.com,99973729,0272934772,NaN,NaN
4,027 257 5622,Owens,C-00088841,Julie,julie.owen@vygr.net,94061035,NaN,NaN,NaN


## 8. Dynamics Contact Type → OneBill Label Map

Dataverse stores the `vgr_contacttypes` field as a multi-select option-set.
The API returns it as a plain comma-separated list of numeric option codes:

    "287790000,287790001"   # two codes: Billing and Technical

We split on commas, strip whitespace, and map each code to the OneBill-side
label. Unknown codes are skipped silently rather than raised, so a new
contact-type added in Dataverse before the map is updated doesn't break
the whole migration.

In [56]:
# Map Dataverse option codes → OneBill contact type labels
CONTACT_TYPE_MAP = {
    "287790000": "Billing",
    "287790001": "Technical",
    "287790002": "Outage - Email",
    "287790009": "Outage - SMS",
    "287790003": "Primary",
    "287790004": "Technical - Data",
    "287790005": "Technical - Voice",
    "287790006": "Commercial",
    "287790008": "Communication",
    "287790007": "Voyager Staff",
}


def parse_contact_types(raw) -> list[str]:
    """
    Parse a vgr_contacttypes value into an ordered list of OneBill labels.

    Handles:
      - None / NaN / empty string  -> []
      - Whitespace around codes
      - Unknown codes (silently skipped rather than raised)
    """
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    codes = [c.strip() for c in str(raw).split(",") if c.strip()]
    return [CONTACT_TYPE_MAP[c] for c in codes if c in CONTACT_TYPE_MAP]

## 10. MySQL Queries

### Account Query

Pulls every Consumer account from the BI view along with derived
first/last-name fields. The query handles a few legacy quirks:

- `firstName` / `lastName` exist on some rows but not others — when missing,
  we split `AccountName` on the last space.
- If both name fields *and* `AccountName` are missing, we fall back to
  `John Doe` so the account can still be created (the OneBill API rejects
  empty names).
- Duplicate `AccountName` values (multiple accounts for the same person)
  are made unique by appending the `AccountCode` in parentheses.
- Empty/null `addr1`, `suburb`, `city`, `postcode` get safe defaults
  rather than failing OneBill's required-field validation.

The output column names are PascalCase to match the rest of the pipeline.

In [72]:
ACCOUNT_QUERY = """
SELECT
    derived.`AccountCode`
    ,derived.`Current_AccountName`
    ,derived.`AccountType`
    ,derived.`OneBill_AccountType`
    ,derived.`Derived_FirstName`
    ,derived.`CreatedDate`
    ,COALESCE(grp.`TotalCount`, 1) AS `TotalCount`
    ,CASE
        WHEN grp.`TotalCount` > 1
        THEN CONCAT(derived.`Derived_FirstName`, ' ', derived.`Derived_LastName`, ' (', derived.`AccountCode`, ')')
        WHEN derived.`Derived_FirstName` = 'John' AND derived.`Derived_LastName` = 'Doe'
        THEN CONCAT(derived.`Derived_FirstName`, ' ', derived.`Derived_LastName`, ' (', derived.`AccountCode`, ')')
        ELSE derived.`Derived_LastName`
    END AS `Unique_LastName`
    ,derived.`EmailAddresses`
    ,derived.`phoneHome`
    ,derived.`mobile`
    ,derived.`Address1`
    ,derived.`Address2`
    ,derived.`Suburb`
    ,derived.`City`
    ,derived.`PostCode`
    ,derived.`DateOfBirth`
FROM (
    SELECT
        `AccountCode`
        ,`AccountName` AS `Current_AccountName`
        ,`AccountType`,
        CASE
            WHEN `AccountType` = 'Actrix Residential' THEN 302
            WHEN `AccountType` = 'Residential'        THEN 303
            WHEN `AccountType` = 'HD Consumer'        THEN 305
            WHEN `AccountType` = 'Staff'              THEN 203
        END AS `OneBill_AccountType`
        ,`CreatedDate`
        ,`_temporary_crmonly_firstname` AS `firstName`
        ,`_temporary_crmonly_lastname` AS `lastName`
        ,CASE
            WHEN `_temporary_crmonly_firstname` IS NOT NULL THEN `_temporary_crmonly_firstname`
            WHEN `AccountName` IS NULL OR `AccountName` = '' THEN 'John'
            ELSE COALESCE(
                NULLIF(TRIM(`_temporary_crmonly_firstname`), ''),
                TRIM(LEFT(
                    TRIM(`AccountName`),
                    LENGTH(TRIM(`AccountName`)) - INSTR(REVERSE(TRIM(`AccountName`)), ' ')
                ))
            )
        END AS `Derived_FirstName`
        ,CASE
            WHEN `_temporary_crmonly_lastname` IS NOT NULL THEN `_temporary_crmonly_lastname`
            WHEN `AccountName` IS NULL OR `AccountName` = '' THEN 'Doe'
            ELSE COALESCE(
                NULLIF(TRIM(`_temporary_crmonly_lastname`), ''),
                TRIM(SUBSTRING_INDEX(TRIM(`AccountName`), ' ', -1))
            )
        END AS `Derived_LastName`
        ,CASE
            WHEN `EmailAddresses` IS NULL OR `EmailAddresses` = '' THEN 'someone@gmail.com'
            ELSE `EmailAddresses`
        END AS `EmailAddresses`
        ,`_temporary_crmonly_phone_home` AS `phoneHome`
        ,`_temporary_crmonly_mobile` AS `mobile`
        ,CASE WHEN `_temporary_crmonly_addr1`    IS NULL OR `_temporary_crmonly_addr1`    = '' THEN '1 Somewhere Place' ELSE `_temporary_crmonly_addr1` END AS `Address1`
        ,`_temporary_crmonly_addr2` AS `Address2`
        ,CASE WHEN `_temporary_crmonly_suburb`   IS NULL OR `_temporary_crmonly_suburb`   = '' THEN 'N/A'               ELSE `_temporary_crmonly_suburb`   END AS `Suburb`
        ,CASE WHEN `_temporary_crmonly_city`     IS NULL OR `_temporary_crmonly_city`     = '' THEN 'Auckland'          ELSE `_temporary_crmonly_city`     END AS `City`
        ,CASE WHEN `_temporary_crmonly_postcode` IS NULL OR `_temporary_crmonly_postcode` = '' THEN 0000                ELSE `_temporary_crmonly_postcode` END AS `Postcode`
        ,`_temporary_crmonly_dob` AS `DateOfBirth`
    FROM bi_datastore.billing_account
        WHERE `AccountType` IN ('Actrix Residential', 'Residential', 'HD Consumer', 'Staff')
		AND `_DataSource` = 'vBill'
) derived
LEFT JOIN (
    SELECT
        TRIM(`AccountName`) AS `AccountName`
        ,COUNT(*)           AS `TotalCount`
        ,MIN(`AccountCode`) AS `Min_AccountCode`
    FROM bi_datastore.billing_account
    WHERE `AccountType` IN ('Actrix Residential', 'Residential', 'HD Consumer', 'Staff')
    AND `_DataSource` = 'vBill'
    GROUP BY TRIM(`AccountName`)
    HAVING COUNT(*) > 1
) grp
    ON TRIM(derived.`Current_AccountName`) = grp.`AccountName`
ORDER BY derived.`Current_AccountName`, derived.`AccountCode`;
"""

engine = create_engine(BI_DATASTORE_URL)
df_accounts = pd.read_sql(ACCOUNT_QUERY, con=engine)

# Making the AccountCode unique by appending a suffix, to avoid conflicts with existing accounts in OneBill during testing. Remove or modify as needed for production.
df_accounts['AccountCode'] = df_accounts['AccountCode'].apply(lambda x: f'{x}_2')

# Remove the next line for a full production run
# df_accounts = df_accounts.head(10)

logger.info(f"Loaded {len(df_accounts):,} accounts from MySQL")
df_accounts.head()

2026-05-13 09:53:19,941 [INFO] Loaded 41,571 accounts from MySQL


,AccountCode,Current_AccountName,AccountType,OneBill_AccountType,Derived_FirstName,CreatedDate,TotalCount,Unique_LastName,EmailAddresses,phoneHome,mobile,Address1,Address2,Suburb,City,Postcode,DateOfBirth
0,94041444_2,None,Actrix Residential,302,Jay,2019-03-29,1,McCartney,jaymccartney@actrix.co.nz,+6463572821,None,15 Rothesay Place,None,N/A,Auckland,0,1970-11-02
1,99960066_2,(MORE INFO) Indu Terese Jos,Residential,303,Indu Terese,2024-05-09,1,Jos,teresejosindu@gmail.com,+64225998452,+64225998452,13b/25 Rutland Street,None,Auckland Central,Auckland,1010,1997-10-26
2,99961066_2,Adetola Adetola,Residential,303,Adetola,2026-01-29,1,Adetola,olan.adetola@gmail.com,+6421766153,+6421766153,6 Selo Street,None,Glen Eden,Waitakere,0602,1993-06-16
3,99967687_2,Amyl forster-wedge,Residential,303,Amyl,2026-04-25,1,forster-wedge,atomic_nucleus15@proton.me,+64.90000000,+6421515750,8,None,Auckland,Auckland,0630,1982-03-15
4,99961709_2,Asmeron Medhanie,Residential,303,Asmeron,2025-01-20,1,Medhanie,amedhanie75@gmail.com,+642108823500,+642108823500,5/96 Richardson Road,None,Mount Albert,Auckland,1025,1997-10-30


### Contact Query

Returns all Active Residential contacts from MySQL

In [58]:
ACCOUNT_QUERY = """
SELECT
    contact.`ContactCode`
    ,contact.`AccountCode`
    ,contact.`ContactType`
    ,contact.`FirstName`
    ,contact.`LastName`
    ,contact.`PhoneWork`
    ,contact.`PhoneMobile`
    ,contact.`EmailAddresses`
    ,contact.`DateOfBirth`
FROM
    bi_curated_views.dynamics_contact contact
LEFT JOIN
    bi_curated_views.reporting_account account
ON
    contact.`AccountCode` = account.`AccountCode`
WHERE
    account.`AccountSegment` = 'Consumer'
ORDER BY
    contact.`AccountCode`;
"""

engine = create_engine(BI_CURATED_VIEWS_URL)
df_sql_contacts = pd.read_sql(ACCOUNT_QUERY, con=engine)

df_sql_contacts = df_sql_contacts.assign(
    BillingContact=True,
    PrimaryContact=True
)

# Remove the next line for a full production run
# df_sql_contacts = df_sql_contacts.head(10)

logger.info(f"Loaded {len(df_sql_contacts):,} contacts from MySQL")
df_sql_contacts.head()

2026-05-13 09:16:17,170 [INFO] Loaded 41,571 contacts from MySQL


,ContactCode,AccountCode,ContactType,FirstName,LastName,PhoneWork,PhoneMobile,EmailAddresses,DateOfBirth,BillingContact,PrimaryContact
0,BILLING-10625002,10625002,Billing contact,Anna,Milroy,None,+6421777157,anna@milroy.kiwi.nz,None,True,True
1,BILLING-12579173,12579173,Billing contact,Cara Tipping Smith t/as Copy Carats,,None,,cara@copycarats.co.nz,None,True,True
2,BILLING-16673533,16673533,Billing contact,Jason,Yee,None,None,yeejase@gmail.com,None,True,True
3,BILLING-21186140,21186140,Billing contact,Lyn-Marie,Harris,+6432818932,+64278072615,None,None,True,True
4,BILLING-24000898,24000898,Billing contact,Kylie,Glenn,+6495706353,+64272442040,kylie@coolawnings.co.nz,None,True,True


## 9. Index Contacts by Account Number

Before the migration starts we group the ~47k contact rows by their parent
account's `accountnumber`. This converts an O(N²) per-account scan into an
O(1) dict lookup during the parallel POST loop.

The `accountnumber` is cast to a string on both sides of the lookup because
MySQL may return it as `int` while Dataverse returns it as `str` — without
the cast, `99961175 != "99961175"` would silently produce empty contact
lists.

In [59]:
df_contacts = pd.concat([df_crm_contacts, df_sql_contacts], ignore_index=True)

# Making the AccountCode unique by appending a suffix, to avoid conflicts with existing accounts in OneBill during testing. Remove or modify as needed for production.
df_contacts['AccountCode'] = df_contacts['AccountCode'].apply(lambda x: f'{x}_2')

# Add default values for any missing columns that the contact creation logic expects
default_values = {
    'LastName': 'Doe',
    'FirstName': 'John',
    'EmailAddresses': 'someone@gmail.com'
}
df_contacts[['LastName', 'FirstName', 'EmailAddresses']] = df_contacts[['LastName', 'FirstName', 'EmailAddresses']].fillna(value=default_values)

df_contacts.head()

,PhoneMobile,LastName,ContactCode,FirstName,EmailAddresses,AccountCode,PhoneWork,ContactType,DateOfBirth,BillingContact,PrimaryContact
0,0212141547,Sibbe,C-00100775,Judy,sibbe@actrix.co.nz,99993083_2,NaN,NaN,NaN,NaN,NaN
1,0211408346,Dunne,C-00092022,Kathryn,katiedunne30@gmail.com,99978495_2,NaN,NaN,NaN,NaN,NaN
2,021951128,Choi,C-00092943,Tina,by09010901@hotmail.com,99969820_2,NaN,NaN,NaN,NaN,NaN
3,+64272934772,Quinn,C-00101851,Laura,laurahopequinn@gmail.com,99973729_2,0272934772,NaN,NaN,NaN,NaN
4,027 257 5622,Owens,C-00088841,Julie,julie.owen@vygr.net,94061035_2,NaN,NaN,NaN,NaN,NaN


In [60]:
def index_contacts_by_account(df: pd.DataFrame) -> dict[str, list[dict]]:
    """
    Group contact rows by accountnumber. Returns {accountnumber_str: [contact_dict, ...]}.

    Drops rows with a null / empty accountnumber, since those cannot be
    matched to a MySQL account.
    """
    by_account: dict[str, list[dict]] = defaultdict(list)
    for _, row in df.iterrows():
        acct = row.get(ACCOUNTNUMBER_COL)
        if pd.isna(acct) or acct in (None, ""):
            continue
        by_account[str(acct)].append(row.to_dict())
    return dict(by_account)

contacts_by_account = index_contacts_by_account(df_contacts)
logger.info(
    f"Indexed {sum(len(v) for v in contacts_by_account.values()):,} contacts "
    f"across {len(contacts_by_account):,} accounts"
)

2026-05-13 09:16:20,006 [INFO] Indexed 47,425 contacts across 41,577 accounts


## 11. OneBill Token Manager (Thread-Safe)

The first version of this pipeline called `get_AccessToken()` inside every
request on every worker thread — thousands of redundant round-trips to the
OAuth endpoint, all serialised through the GIL. This class caches one token
across all threads and refreshes it proactively ~100s before expiry.

The lock is held only during refresh; the common case (token still valid)
returns immediately under the lock without any I/O. If two threads happen
to arrive while the token is stale, only one performs the refresh — the
other waits at the lock and then sees the freshly-cached token.

In [61]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:
            if datetime.now() >= self._expires_at:
                self._refresh()
            return self._token

    def _refresh(self) -> None:
        logger.info("Refreshing OneBill OAuth token...")
        token_data = {
            "grant_type":    "password",
            "client_id":     os.environ["CLIENT_ID"],
            "client_secret": os.environ["CLIENT_SECRET"],
            "username":      os.environ["API_USERNAME"],
            "password":      os.environ["API_PASSWORD"],
        }
        response = requests.post(
            ONEBILL_TOKEN_URL,
            data=token_data,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload["access_token"]
        ttl = payload.get("expires_in", TOKEN_TTL_SECONDS)
        # Refresh 100s early to absorb clock skew + in-flight requests
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)
        logger.info(
            "Token refreshed; valid until %s",
            self._expires_at.strftime("%H:%M:%S"),
        )


token_manager = TokenManager()

## 12. Payload Builders — Account + Contacts

Two helpers feed the main payload builder:

- **`build_contact_block(contact)`** — takes a single Dynamics contact dict
  and returns one OneBill-shaped contact entry. `communicationPoint`
  emits Email/Mobile/Phone in that order, skipping any null values.
  `contactAttributes` is only emitted if at least one type was parsed.
- **`build_account_payload(row, contacts_by_account)`** — composes the
  full account-creation JSON. The first element of `contact[]` is the
  account-level person from the MySQL row (no `contactAttributes`). Every
  Dynamics contact whose `accountnumber` matches the account is appended
  after, each with its own `contactAttributes`.

Note: the literal `"visiblity"` typo in `contactAttributes` is preserved
to match the OneBill schema as documented. If that turns out to be a
documentation error rather than a real schema field, change it to
`"visibility"` here in one place.

In [62]:
def build_contact_block(contact: dict) -> dict:
    """
    Build a single contact entry from a Dynamics contact row.

    Includes Email / Mobile / Phone communication points (nulls skipped),
    and a contactAttributes block listing all assigned Dynamics contact
    types via the multi-value custom attribute.
    """
    # --- Communication points: emit only the channels that have a value
    comm_points = []
    if contact.get("EmailAddresses"):
        comm_points.append({"type": "Email", "value": contact["EmailAddresses"]})
    if contact.get("MobilePhone"):
        comm_points.append({"type": "Phone", "value": contact["MobilePhone"]})

    block = {
        "firstName":          contact.get("FirstName"),
        "lastName":           contact.get("LastName"),
        "primaryContact":     False,
        "billingContact":     False,
        "communicationPoint": comm_points,
    }

    # --- Contact-type attribute (multi-value)
    types = parse_contact_types(contact.get("ContactType"))
    if types:
        block["contactAttributes"] = [{
            "id":                      "160203",
            "key":                     "Dynamics Contact Types",
            "value":                   types[0],           # primary label
            "sequence":                0,
            "aggregator":              0,
            "visiblity":               1,                  # OneBill schema spelling
            "multipleEntriesConfig":   "ENABLED",
            "attributeValuesInfo": {
                "associateValues": [
                    {"value": t, "sequence": i + 1}
                    for i, t in enumerate(types)
                ],
            },
        }]

    return block


def serialize_date(value, fmt: str | None = None):
    """
    Format a date/datetime/string into the shape OneBill expects.

    - `fmt=None` → ISO 8601 (used for activationStartDate)
    - `fmt='%d/%m/%Y'` → NZ-style for accountAttribute Date Of Birth
    """
    if value is None:
        return None
    if hasattr(value, "isoformat"):
        return value.strftime(fmt) if fmt else value.isoformat()
    if fmt:
        try:
            return datetime.strptime(str(value), "%Y-%m-%d").strftime(fmt)
        except ValueError:
            return str(value)
    return str(value)


def build_account_payload(row: pd.Series, contacts_by_account: dict[str, list[dict]]) -> str:
    """
    Build the full OneBill account-creation payload, including:

      - The account-level contact (from MySQL — no contactAttributes).
      - All Dynamics contacts matching this account's number, each with
        their own contactAttributes block.
    """
    # Replace pandas NaN with Python None so json.dumps doesn't choke
    row = row.where(pd.notna(row), None).to_dict()

    # --- Account-level contact (always present)
    contact_list: list[dict] = [{
        "firstName": row["Derived_FirstName"],
        "lastName":  row["Unique_LastName"],
        "communicationPoint": [
            {"type": "Email", "value": row["EmailAddresses"]},
            {"type": "Phone", "value": row["mobile"]},
        ],
    }]

    # --- Any additional Dynamics contacts on the same account
    # Cast to str so int (MySQL) and str (Dataverse) compare equal
    extra_contacts = contacts_by_account.get(str(row["AccountCode"]), [])
    contact_list.extend(build_contact_block(c) for c in extra_contacts)

    payload = {
        "accountNumber":       row["AccountCode"],
        "accountingDisplayName": row["Current_AccountName"],
        "accountType":         "1001",
        "accountSubType":      row["OneBill_AccountType"],
        "activationStartDate": serialize_date(row["CreatedDate"]),
        "address": [{
            "addLine1":        row["Address1"],
            "addLine2":        row["Address2"],
            "city":            row["City"],
            "state":           None,
            "country":         "New Zealand",
            "zip":             row["Postcode"],
            "defaultShipping": "true",
            "defaultBilling":  "true",
        }],
        "contact": contact_list,
        "accountAttribute": [{
            "key":   "Date Of Birth",
            "value": serialize_date(row["DateOfBirth"], fmt="%d/%m/%Y"),
        }],
    }

    return json.dumps(payload)

## 13. POST to OneBill

A single account creation. The shared `requests.Session` is reused across
all threads to take advantage of HTTP connection pooling. The bearer token
is read from `token_manager` per request — cheap when the token is valid,
and automatically rotated when it isn't.

`validationResponse.successful = False` in the response body is treated as
a failure even when the HTTP status is 200, because OneBill returns
business-logic errors that way (e.g. "Account number already exists").

In [63]:
def create_onebill_account(session: requests.Session, base_url: str, payload: str) -> dict:
    """POST one account to OneBill. Raises ValueError on validation failure."""
    url = f"{base_url}/rest/SubscriberService/v1/subscriber"
    headers = {"Authorization": f"Bearer {token_manager.get_token()}"}

    response = session.post(url, headers=headers, data=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    validation = data.get("validationResponse", {})
    if not validation.get("successful", True):
        errors   = validation.get("validationErrorInfo", [])
        messages = "; ".join(e.get("message", "") for e in errors)
        raise ValueError(messages)

    return data

## 14. Per-Row Worker (with Profiling)

Each worker invocation handles one account: build the payload, POST it,
record the outcome. Build time and network time are recorded separately so
performance bottlenecks can be diagnosed from the results DataFrame.

`contacts_by_account` is passed in explicitly rather than captured by
closure, which makes the function easier to test in isolation.

In [64]:
def migrate_row(
    row: pd.Series,
    session: requests.Session,
    contacts_by_account: dict[str, list[dict]],
) -> dict:
    """Build + POST a single account, returning a result dict for the summary."""
    account_code = row["AccountCode"]
    first_name   = row["Derived_FirstName"]
    last_name    = row["Unique_LastName"]

    # --- Build the payload (cheap)
    t0      = time.perf_counter()
    payload = build_account_payload(row, contacts_by_account)
    t_build = time.perf_counter() - t0

    # --- POST (network)
    t_net = 0.0
    try:
        t1 = time.perf_counter()
        response = create_onebill_account(session, ONEBILL_BASE_URL, payload)
        t_net = time.perf_counter() - t1

        onebill_id = response.get("accountId", "unknown")
        logger.info(
            f"  [OK] {account_code} (OneBill id={onebill_id}) — "
            f"build={t_build*1000:.0f}ms net={t_net*1000:.0f}ms"
        )
        status, error = "success", None

    except Exception as e:
        # If we failed before issuing the request, t_net stays 0
        if "t1" in locals():
            t_net = time.perf_counter() - t1
        logger.error(f"  [FAIL] {account_code} — {e}")
        status, error = "failed", str(e)

    return {
        "account_code":     account_code,
        "first_name":       first_name,
        "last_name":        last_name,
        "status":           status,
        "error":            error,
        "elapsed_build_ms": round(t_build * 1000, 1),
        "elapsed_net_ms":   round(t_net   * 1000, 1),
    }

## 15. Parallel Migration Loop

Fans out the account list across `MAX_WORKERS` threads using
`ThreadPoolExecutor`. The HTTP session is shared (with a pool sized to
match worker count) so connections are reused rather than re-established
for every request.

Progress is logged every 50 records. The final profiling summary surfaces
throughput, mean and P95 network latency, and the build/net split so it's
obvious where any slowdown is.

In [65]:
def migrate(
    df: pd.DataFrame,
    contacts_by_account: dict[str, list[dict]],
    max_workers: int = MAX_WORKERS,
) -> pd.DataFrame:
    """Migrate every account in df to OneBill in parallel."""
    # --- Shared session with a connection pool sized to the worker count
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers,
    )
    session.mount("https://", adapter)
    session.headers.update({
        "proxy_accountNumber": ONEBILL_PROXY_ACCT,
        "Content-Type":        "application/json",
    })

    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    results: list[dict] = []

    logger.info(f"Starting migration of {total:,} records with {max_workers} workers...")
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(migrate_row, row, session, contacts_by_account): row["AccountCode"]
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())

            if i % 50 == 0 or i == total:
                ok   = sum(1 for r in results if r["status"] == "success")
                fail = sum(1 for r in results if r["status"] == "failed")
                logger.info(f"Progress: {i}/{total} — {ok} ok, {fail} failed")

    wall_elapsed = time.perf_counter() - wall_start
    results_df = pd.DataFrame(results)
    success = (results_df["status"] == "success").sum()
    failed  = (results_df["status"] == "failed").sum()

    logger.info(
        f"Migration done in {wall_elapsed:.1f}s — "
        f"{success} succeeded, {failed} failed. (log: {log_filename})"
    )

    # --- Profiling summary
    print("\n=== Profiling Summary ===")
    print(f"Total wall time:          {wall_elapsed:.1f}s")
    print(f"Throughput:               {total / wall_elapsed:.1f} accounts/s")
    print(f"Avg build time per row:   {results_df['elapsed_build_ms'].mean():.1f}ms")
    print(f"Avg network time per row: {results_df['elapsed_net_ms'].mean():.1f}ms")
    print(f"Max network time:         {results_df['elapsed_net_ms'].max():.1f}ms")
    print(f"P95 network time:         {results_df['elapsed_net_ms'].quantile(0.95):.1f}ms")
    print("=========================")

    return results_df

## 16. Run the Migration

Kicks off the full parallel run. With ~50k accounts at 20 workers and
typical OneBill sandbox latency this should take a couple of minutes,
network permitting.

In [66]:
results_df = migrate(df_accounts, contacts_by_account)

failures = results_df[results_df["status"] == "failed"]
print(f"\nFailed rows ({len(failures):,}):")
failures

2026-05-13 09:16:20,129 [INFO] Starting migration of 10 records with 20 workers...
2026-05-13 09:16:20,135 [INFO] Refreshing OneBill OAuth token...
2026-05-13 09:16:21,139 [INFO] Token refreshed; valid until 09:27:10
2026-05-13 09:16:37,739 [INFO]   [OK] 99961709_2 (OneBill id=unknown) — build=1ms net=17570ms
2026-05-13 09:16:37,863 [INFO]   [OK] 94041444_2 (OneBill id=unknown) — build=1ms net=17727ms
2026-05-13 09:16:38,026 [INFO]   [OK] 99964529_2 (OneBill id=unknown) — build=0ms net=17852ms
2026-05-13 09:16:38,079 [INFO]   [OK] 99965756_2 (OneBill id=unknown) — build=0ms net=17907ms
2026-05-13 09:16:38,103 [INFO]   [OK] 99969085_2 (OneBill id=unknown) — build=0ms net=17927ms
2026-05-13 09:16:38,891 [INFO]   [OK] 99960066_2 (OneBill id=unknown) — build=1ms net=18732ms
2026-05-13 09:16:38,905 [INFO]   [OK] 99968511_2 (OneBill id=unknown) — build=0ms net=18730ms
2026-05-13 09:16:38,906 [INFO]   [OK] 99961066_2 (OneBill id=unknown) — build=0ms net=18745ms
2026-05-13 09:16:38,909 [INFO] 


=== Profiling Summary ===
Total wall time:          18.8s
Throughput:               0.5 accounts/s
Avg build time per row:   0.7ms
Avg network time per row: 18267.9ms
Max network time:         18747.9ms
P95 network time:         18746.4ms

Failed rows (0):


,account_code,first_name,last_name,status,error,elapsed_build_ms,elapsed_net_ms


## 17. Export Failures for Investigation

Saves every failure to a timestamped CSV so the migration team can sort
through them — re-run candidates, true rejects, schema issues, etc.

In [67]:
out_path = f'Failed_Migrations_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
failures.to_csv(out_path, index=False)
print(f"Wrote {len(failures):,} failures to {out_path}")

Wrote 0 failures to Failed_Migrations_20260513_091638.csv


## 18. Failed Account → Contacts Lookup

For every failed account, pull its associated Dynamics contacts back out
of `df_contacts` so you can inspect them side-by-side with the failure
reason. Useful for spotting patterns like:

- The same bad email format across all of an account's contacts.
- A `vgr_contacttypes` value that didn't map to anything in `CONTACT_TYPE_MAP`.
- Contacts with `firstname`/`lastname` that OneBill rejected as too long.

Two outputs are produced:

1. **`failures_with_contacts`** — a long-format DataFrame, one row per
   `(failed_account, contact)` pair, including the failure reason. Failed
   accounts that have **no** Dynamics contacts also appear with the
   contact fields left null, so you don't accidentally drop them when
   filtering.
2. **A CSV export** alongside the failures file for offline review.

In [68]:
def lookup_failed_contacts(failures_df: pd.DataFrame, df_contacts: pd.DataFrame) -> pd.DataFrame:
    """
    Join failed accounts to their Dynamics contacts.

    Returns a long-format DataFrame with one row per (failed_account, contact)
    pair. Accounts with no matching contacts are kept as a single row with
    the contact columns null, via a LEFT join.
    """
    if failures_df.empty:
        return pd.DataFrame()

    # The contacts DataFrame stores accountnumber under the aliased column —
    # rename to a tidy key for the merge, and cast both sides to string so
    # int (MySQL) and str (Dataverse) compare equal.
    contacts_lookup = df_contacts.rename(columns={ACCOUNTNUMBER_COL: "AccountCode"}).copy()
    contacts_lookup["AccountCode"] = contacts_lookup["AccountCode"].astype(str)

    fails = failures_df.copy()
    fails["AccountCode"] = fails["AccountCode"].astype(str)

    # LEFT join keeps failed accounts that have zero contacts in Dynamics —
    # without this they'd silently disappear from the output.
    joined = fails.merge(
        contacts_lookup[[
            "AccountCode", "FirstName", "LastName",
            "EmailAddresses", "MobilePhone", "WorkPhone", "ContactTypes",
        ]],
        on="AccountCode",
        how="left",
        suffixes=("_account", "_contact"),
    )

    return joined


failures_with_contacts = lookup_failed_contacts(failures, df_contacts)

# Quick at-a-glance summary
if not failures_with_contacts.empty:
    n_failed_accounts   = failures_with_contacts["AccountCode"].nunique()
    n_contact_rows      = failures_with_contacts["ContactId"].notna().sum()
    n_accounts_no_contacts = (
        failures_with_contacts.groupby("AccountCode")["ContactId"]
        .apply(lambda s: s.isna().all())
        .sum()
    )
    print(
        f"{n_failed_accounts:,} failed accounts → "
        f"{n_contact_rows:,} associated contacts "
        f"({n_accounts_no_contacts:,} had no Dynamics contacts at all)"
    )

failures_with_contacts.head(20)

""


In [69]:
# Persist for offline review
fwc_path = f'Failed_Migrations_with_Contacts_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
failures_with_contacts.to_csv(fwc_path, index=False)
print(f"Wrote {len(failures_with_contacts):,} failure+contact rows to {fwc_path}")

Wrote 0 failure+contact rows to Failed_Migrations_with_Contacts_20260513_091639.csv


## 19. Failure Summary by Error Message

Groups failures by their error text so you can see which problem is hitting
the most accounts. Useful for triaging — fix the most common error first,
re-run, repeat.

In [70]:
if not failures.empty:
    error_summary = (
        failures.groupby("error")
        .agg(count=("account_code", "size"),
             example_account=("account_code", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
    print(f"Distinct error messages: {len(error_summary):,}")
    error_summary
else:
    print("No failures to summarise.")
    error_summary = pd.DataFrame()
error_summary

No failures to summarise.


""


In [71]:
print(len(df_accounts))

10
